# Fase 1: Análisis Exploratorio de Datos (EDA)
## Detección y Predicción de Desgaste de Herramienta en Fresadora CNC

**Autor:** Íñigo Guzmán  
**Dataset:** University of Michigan SMART Lab - CNC Mill Tool Wear Dataset  

### Objetivo del Análisis
Comprender las dinámicas de corte y el comportamiento electro-mecánico de los servomotores (ejes X, Y, Z) y del husillo principal (S1) en condiciones de corte con herramientas afiladas (*unworn*) vs desgastadas (*worn*).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

data_dir = Path("../data/raw")
df_train = pd.read_csv(data_dir / "train.csv")
df_train.head(10)

### 1. Balance de Clases y Metadatos
El dataset cuenta con 18 experimentos supervisados:
- **8 experimentos con herramienta sin desgaste (unworn)**
- **10 experimentos con herramienta desgastada (worn)**

Se varían controladamente la velocidad de avance (*feedrate* entre 3 y 20 mm/s) y la presión del pistón neumático (*clamp_pressure* entre 2.5 y 4.0 bar).

In [ ]:
print("Distribución de condición de la herramienta:")
print(df_train["tool_condition"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(
    data=df_train,
    x="tool_condition",
    palette=["#2b5c8f", "#d95f02"],
    hue="tool_condition",
    legend=False,
)
plt.title("Distribución de Clases (Experimentos)")
plt.xlabel("Condición de la Herramienta")
plt.ylabel("Recuento")
plt.show()

### 2. Señales de Telemetría de Alta Frecuencia (100 ms)
Cada experimento contiene señales continuas de 48 variables de estado registradas cada 100 ms.
Las señales críticas identificadas en el mecanizado son:
- `S1_OutputCurrent` / `S1_CurrentFeedback`: Esfuerzo de par motor y fricción rotacional del husillo.
- `Z1_ActualAcceleration` / `Z1_CurrentFeedback`: Fuerza axial vertical; al perder filo los labios de la fresa, la herramienta tiende a empujar el material en vez de cortarlo limpiamente, incrementando la resistencia en Z.

In [ ]:
df_exp1 = pd.read_csv(data_dir / "experiment_01.csv")  # Unworn
df_exp6 = pd.read_csv(data_dir / "experiment_06.csv")  # Worn

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(
    df_exp1.index * 0.1,
    df_exp1["S1_OutputCurrent"],
    label="Exp 01 (Unworn)",
    color="#2b5c8f",
    alpha=0.8,
)
ax.plot(
    df_exp6.index * 0.1,
    df_exp6["S1_OutputCurrent"],
    label="Exp 06 (Worn)",
    color="#d95f02",
    alpha=0.8,
)
ax.set_title("Comparación Temporal de Corriente del Husillo (S1)")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Corriente de Salida (A)")
ax.legend()
plt.show()

### 3. Fases de Mecanizado (`Machining_Process`)
Las fases `Starting`, `Prep` y `Repositioning` corresponden a movimientos de posicionamiento en vacío sin contacto de corte.
Las fases activas de corte corresponden a las pasadas de fresado (`Layer 1 Up`, `Layer 1 Down`, etc.). Para el entrenamiento del modelo de ML, aislar las fases de corte activas reduce el ruido de posicionamiento.

In [ ]:
print("Fases de mecanizado presentes:")
print(df_exp1["Machining_Process"].value_counts())

df_cut1 = df_exp1[df_exp1["Machining_Process"].str.startswith("Layer", na=False)]
df_cut6 = df_exp6[df_exp6["Machining_Process"].str.startswith("Layer", na=False)]

print(
    f"Media de corriente S1 en corte Exp 01 (Nueva): {df_cut1['S1_OutputCurrent'].mean():.2f} A"
)
print(
    f"Media de corriente S1 en corte Exp 06 (Desgastada): {df_cut6['S1_OutputCurrent'].mean():.2f} A"
)

### 4. Conclusiones y Guía para Feature Engineering (Fase 2)
1. **Riesgo crítico de Data Leakage:** Jamás realizar un `train_test_split` aleatorio por filas individuales. Las muestras contiguas de un mismo experimento están autocorrelacionadas temporalmente. Se debe utilizar **`GroupKFold`** agrupando obligatoriamente por el identificador del experimento (`No`).
2. **Extracción de Features Agregadas:** Por cada experimento extraeremos métricas estadísticas en las fases activas de corte: media, desviación típica, valor RMS (Root Mean Square), picos máximos y rango intercuartílico de las corrientes del husillo y de los ejes X, Y y Z.